# MIMYR：MERFISH 五切片的三折预训练权重预测

**本 notebook 只做冻结权重推理，不训练、不微调、不更换模型头或词表。**
参考 `RC_DeepSpatial/examples/run_merfish.py`：依次留出 `merfish_1、2、3`，seed 为 42；
每折的相邻参考分别为 `[0, 2]`、`[1, 3]`、`[2, 4]`。三折使用同一套作者权重，属于三次留出预测，
不是重新训练三套模型的交叉验证。

推理设置取自本仓库 `run.md`、`main.py`、`inference.py`、`models/combined_model.py`：
位置、类别、表达三阶段均使用模型；保留 70 个扩散步、FP32、原表达生成批量 1500、
`max_new_tokens=500`、`top_k=5` 和 KV cache。原论文完整数据集的划分不适用于这五张切片；
这里只替换数据入口与三折划分，不把本数据集分数称为论文原表数值的复现。

## 输入的实际限制

已检查的这五张原始数据只有局部二维 `obsm['spatial']`，**没有 Allen CCF 配准坐标**。
作者预训练位置和类别模型使用 CCF mm 坐标；仅除以 1000、平移到脑图谱范围或补上相对深度，
都不能替代真实配准。本 notebook 会先审查原始数据；缺少配准时在加载模型前停止。
这时完成的是数据检查，不能把它报告为三折预测已经完成。

原数据的 `cell_class` 与模型的 5274 类 `cluster` 不是同一个标签体系。
预测保留作者类别，主评估使用原脚本的表达 soft Spearman 和 soft F1，基于实测与模型基因的交集；
不会把 `cell_class` 直接编码成作者 cluster，也不默认计算二者的类别准确率。

为沿用原仓库生成流程，模型获得留出切片的**目标细胞数量、空间边界及切面位置**，
不获得其表达或细胞标签。这与完全不使用目标几何的盲重建协议不同，比较 RC DeepSpatial 时需注明。

## 1. 在 Linux 服务器复制原始数据

在服务器终端执行以下命令。只复制五个指定文件，自动查找嵌套解压目录；保留已有文件并核对内容。

```bash
set -euo pipefail
src=/3250608056/code/DeepSpatial/data/merfish_mouse_hypothalamus
dst=/3250608056/code/MIMYR/data/merfish_mouse_hypothalamus/raw
mkdir -p "$dst"
for i in 0 1 2 3 4; do
  mapfile -d '' matches < <(find "$src" -type f -name "merfish_${i}.h5ad" -print0)
  if [ "${#matches[@]}" -ne 1 ]; then
    printf 'merfish_%s.h5ad 应当且只能找到一份，实际为 %s\n' "$i" "${#matches[@]}" >&2
    exit 1
  fi
  target="$dst/merfish_${i}.h5ad"
  if [ ! -e "$target" ]; then cp -- "${matches[0]}" "$target"; fi
  cmp -- "${matches[0]}" "$target"
done
ls -lh "$dst"/merfish_[0-4].h5ad
```

将本 notebook 放在服务器 `MIMYR/notebooks/`，选择安装好项目依赖的 Python 内核。
Jupyter 未安装时可在该环境执行 `python -m pip install notebook ipykernel`。

配准坐标有两种输入方式，均需来自实际的 CCF 配准：

1. 五张 `.h5ad` 已有 `obs['x_ccf']、obs['y_ccf']、obs['z_ccf']`，单位为 mm。
2. 原始文件不变，另放 `MIMYR/data/merfish_mouse_hypothalamus/ccf_coordinates.csv`，包含：
   `slice_id,cell_id,x_ccf,y_ccf,z_ccf`。`slice_id` 为 0–4，`cell_id` 必须匹配对应文件的 `obs_names`。

这里使用原 `rq1` 的冠状切面约定：`x_ccf` 为切片轴，内部 `aligned_spatial=[z_ccf, y_ccf, x_ccf]`。
如果配准输出为其他轴向，应先确认对应模型与实验设置，不能仅交换列名冒充该设置。

In [ ]:
import os
import sys
from pathlib import Path

GPU_ID = '0'  # 服务器上的物理 GPU；修改后重启内核
if 'torch' in sys.modules:
    raise RuntimeError('请先重启内核，再从本单元开始运行，以正确选择 GPU。')
if GPU_ID is not None:
    os.environ['CUDA_VISIBLE_DEVICES'] = GPU_ID

candidates = [Path.cwd(), *Path.cwd().parents]
PROJECT = next((p.resolve() for p in candidates
                if (p / 'main.py').is_file() and (p / 'models/combined_model.py').is_file()), None)
if PROJECT is None:
    raise FileNotFoundError('请从 MIMYR 项目或其 notebooks 目录打开本 notebook。')
os.chdir(PROJECT)
sys.path.insert(0, str(PROJECT))

DATA_DIR = PROJECT / 'data/merfish_mouse_hypothalamus'
RAW_DIR = DATA_DIR / 'raw'
CCF_CSV = DATA_DIR / 'ccf_coordinates.csv'
METADATA_DIR = PROJECT / 'models/generative_transformer/metadata'
OUTPUT = PROJECT / 'results/merfish_pretrained_3fold'
HOLDOUTS = [1, 2, 3]
SEED = 42
RESUME = True  # 仅复用配置、输入、源码及权重哈希均一致且完成的折

print('项目:', PROJECT)
print('数据:', RAW_DIR)
print('输出:', OUTPUT)

In [ ]:
import copy
import gc
import hashlib
import json
import pickle
import platform
import time
import urllib.request
from types import SimpleNamespace

import anndata as ad
import numpy as np
import pandas as pd
import torch
from scipy import sparse
from IPython.display import display

CHECKPOINTS = {
    'location': PROJECT / 'model_checkpoints/smoothtune_conditional_ddpm_2d_checkpoint_400.pt',
    'cluster': PROJECT / 'model_checkpoints/best_model_rq1.pt',
    'expression': PROJECT / 'model_checkpoints/TG-base4_epoch4_model.pt',
}
CFG = {
    'run_mode': 'inference', 'data_mode': 'rq1', 'data_label': 'cluster',
    'location_inference_type': 'model', 'cluster_inference_type': 'model',
    'expression_inference_type': 'model', 'kde_bandwidth': 0.01,
    'guidance_signal': 0.01, 'batch_size': 1024, 'device': 'cuda',
    'metrics': ['soft_spearman_correlation', 'soft_f1'],
    'metric_sampling': 100, 'full_gene_panel': True,
    'meta_info': '4hierarchy_metainfo_mouse_geneunion2_DAG.pt',
    'location_model_checkpoint': str(CHECKPOINTS['location']),
    'cluster_model_checkpoint': str(CHECKPOINTS['cluster']),
    'expression_model_checkpoint': str(CHECKPOINTS['expression']),
}

def sha256(path):
    h = hashlib.sha256()
    with Path(path).open('rb') as f:
        for chunk in iter(lambda: f.read(4 * 1024 * 1024), b''):
            h.update(chunk)
    return h.hexdigest()

def git_blob_sha(data):
    return hashlib.sha1(b'blob ' + str(len(data)).encode() + b'\0' + data).hexdigest()

def write_json(path, value):
    path = Path(path)
    temporary = path.with_suffix(path.suffix + '.tmp')
    temporary.write_text(json.dumps(value, ensure_ascii=False, indent=2, allow_nan=False), encoding='utf-8')
    temporary.replace(path)

def fold_definition(heldout):
    if heldout not in HOLDOUTS:
        raise ValueError('只运行内部切片 1、2、3。')
    return {'heldout': heldout, 'visible': [i for i in range(5) if i != heldout],
            'reference': [heldout - 1, heldout + 1], 'seed': SEED}

display(pd.DataFrame([fold_definition(i) for i in HOLDOUTS]))
print('PyTorch:', torch.__version__, '| CUDA:', torch.cuda.is_available())
if torch.cuda.is_available():
    props = torch.cuda.get_device_properties(0)
    print('GPU:', props.name, '| 显存 GiB:', round(props.total_memory / 2**30, 2))
print('推理配置:', json.dumps(CFG, ensure_ascii=False, indent=2))

## 2. 核对权重和原始元信息

不下载或替换模型权重。原代码还使用 `hierarchy.pkl`，本仓库初始缺少该文件；
下一个单元只在它缺失时，从作者官方仓库的固定提交下载该元数据并校验 Git blob 哈希。
下载来源：[作者固定提交](https://github.com/gkrieg/mimyr/tree/f96115d978ed619add1bb65b21a85b05ba41c13f)。
本地原有的表达词表和 `id_to_subclass.pkl` 与该提交中的文件一致；运行时再次核对。

In [ ]:
missing_weights = [str(p) for p in CHECKPOINTS.values() if not p.is_file()]
if missing_weights:
    raise FileNotFoundError('先把已有权重复制到服务器对应目录：\n' + '\n'.join(missing_weights))

META_PATH = METADATA_DIR / CFG['meta_info']
ID_PATH = PROJECT / 'id_to_subclass.pkl'
hierarchy_path = METADATA_DIR / 'hierarchy.pkl'
expected_blobs = {
    META_PATH: '0c0b63d501c42f4a2febf65ddf5a1c858f0ca99f',
    ID_PATH: '266bf5fde2b6c89265a0b2a638c42c06cab65ecf',
    hierarchy_path: 'c54536a034ae8c6d2b03f14866aacc22232e1921',
}
for path in [META_PATH, ID_PATH]:
    if not path.is_file() or git_blob_sha(path.read_bytes()) != expected_blobs[path]:
        raise ValueError(f'缺少或不匹配的原始元数据：{path}')
if not hierarchy_path.exists():
    url = ('https://raw.githubusercontent.com/gkrieg/mimyr/'
           'f96115d978ed619add1bb65b21a85b05ba41c13f/'
           'src/mimyr/models/generative_transformer/metadata/hierarchy.pkl')
    with urllib.request.urlopen(url, timeout=60) as response:
        content = response.read()
    if git_blob_sha(content) != expected_blobs[hierarchy_path]:
        raise ValueError('下载的 hierarchy.pkl 校验失败。')
    hierarchy_path.write_bytes(content)
if git_blob_sha(hierarchy_path.read_bytes()) != expected_blobs[hierarchy_path]:
    raise ValueError('hierarchy.pkl 与固定作者版本不匹配。')

META = torch.load(META_PATH, map_location='cpu', weights_only=False)
ID_TO_LABEL = pickle.loads(ID_PATH.read_bytes())
HIERARCHY = pickle.loads(hierarchy_path.read_bytes())
if len(ID_TO_LABEL) != 5274 or set(ID_TO_LABEL) != set(range(5274)):
    raise ValueError('预训练细胞类别映射不完整。')
for label in ID_TO_LABEL.values():
    if ('cluster', label) not in HIERARCHY:
        raise ValueError(f'缺少预测类别的层级映射：{label}')
EDGES = {c: pickle.loads((METADATA_DIR / f'edges_{c}.pkl').read_bytes()) for c in 'xyz'}
print('模型词表:', len(META['token_set']), '| 模型基因:', len(META['gene_set']),
      '| 预训练类别:', len(ID_TO_LABEL))

## 3. 数据审查与 CCF 输入检查

审查会显示每张切片的细胞数量、实测基因数、与模型基因的交集和坐标状态。
**不会把局部空间坐标改造成 CCF，也不会把 cell_class 当作 cluster。**
若原始文件已有真实 CCF 列，下方直接读取；否则按 cell ID 合并配准 CSV。
CCF 数值范围检查只能发现明显的格式/单位错误，不能证明配准的生物学正确性。

In [ ]:
CCF_COLS = ['x_ccf', 'y_ccf', 'z_ccf']

def ccf_coordinates(a, slice_id, sidecar=None):
    if all(c in a.obs for c in CCF_COLS):
        coords = a.obs[CCF_COLS].to_numpy(dtype=float)
    elif sidecar is not None:
        needed = {'slice_id', 'cell_id', *CCF_COLS}
        if not needed.issubset(sidecar.columns):
            raise ValueError('CCF CSV 必须包含 slice_id,cell_id,x_ccf,y_ccf,z_ccf。')
        rows = sidecar.loc[sidecar['slice_id'].astype(int) == slice_id].copy()
        rows['cell_id'] = rows['cell_id'].astype(str)
        if rows['cell_id'].duplicated().any():
            raise ValueError(f'切片 {slice_id} 的 CCF cell_id 重复。')
        rows = rows.set_index('cell_id')
        if not a.obs_names.astype(str).isin(rows.index).all():
            raise ValueError(f'切片 {slice_id} 的 CCF 坐标没有覆盖全部 cell ID。')
        coords = rows.loc[a.obs_names.astype(str), CCF_COLS].to_numpy(dtype=float)
    else:
        raise ValueError(f'merfish_{slice_id} 缺少真实 CCF 配准坐标；仅有 spatial 不能使用图谱权重。')
    if coords.shape != (a.n_obs, 3) or not np.isfinite(coords).all():
        raise ValueError('CCF 坐标必须是有限的 N×3 数组，单位 mm。')
    for j, c in enumerate('xyz'):
        edges = np.asarray(EDGES[c])
        if (coords[:, j] < edges.min()).any() or (coords[:, j] > edges.max()).any():
            raise ValueError(f'{c}_ccf 超出模型坐标分箱范围；检查坐标顺序、单位与配准结果。')
    return coords

def read_raw(slice_id):
    path = RAW_DIR / f'merfish_{slice_id}.h5ad'
    if not path.is_file():
        raise FileNotFoundError(f'请先执行复制命令：{path}')
    a = ad.read_h5ad(path)
    if not a.n_obs or not a.obs_names.is_unique or not a.var_names.is_unique:
        raise ValueError(f'{path.name} 需要非空数据及唯一 cell/gene ID。')
    values = a.X.data if sparse.issparse(a.X) else np.asarray(a.X)
    if not np.isfinite(values).all() or (values < 0).any():
        raise ValueError(f'{path.name} 的表达矩阵需要有限、非负。')
    if not set(a.var_names).intersection(META['gene_set']):
        raise ValueError(f'{path.name} 没有模型可评估的共同基因。')
    return a

sidecar = pd.read_csv(CCF_CSV, dtype={'cell_id': str}) if CCF_CSV.is_file() else None
audit_rows, ccf_errors, ccf_depths = [], [], []
for i in range(5):
    a = read_raw(i)
    row = {'slice': i, 'cells': a.n_obs, 'measured_genes': a.n_vars,
           'model_overlap_genes': len(set(a.var_names).intersection(META['gene_set'])),
           'has_cell_class': 'cell_class' in a.obs, 'ccf_ready': False}
    try:
        xyz = ccf_coordinates(a, i, sidecar)
        row.update(ccf_ready=True, x_ccf_mean_mm=float(xyz[:, 0].mean()))
        ccf_depths.append(float(xyz[:, 0].mean()))
    except ValueError as exc:
        ccf_errors.append(str(exc))
    audit_rows.append(row)
    del a
audit = pd.DataFrame(audit_rows)
display(audit)
OUTPUT.mkdir(parents=True, exist_ok=True)
audit.to_csv(OUTPUT / 'input_audit.csv', index=False)
if ccf_errors:
    raise RuntimeError('当前只能完成数据审查，尚不能运行有效的预训练预测。\n' + '\n'.join(ccf_errors)
                       + '\n请提供真实 CCF 配准结果后重跑本单元。原文件和模型均未修改。')
delta = np.diff(ccf_depths)
if not (np.all(delta > 0) or np.all(delta < 0)):
    raise ValueError('文件 0–4 的 CCF 切面位置应按同一方向排序；检查切片顺序与所选冠状模型。')

## 4. 输入连接的自检

这是工程检查，不训练模型，也不产生论文指标。验证三折的目标不进入参考，以及没有 CCF 时确实拒绝预测。

In [ ]:
def input_self_check():
    for h in HOLDOUTS:
        fold = fold_definition(h)
        assert h not in fold['visible'] and h not in fold['reference']
        assert len(fold['visible']) == 4 and fold['reference'] == [h - 1, h + 1]
    dummy = ad.AnnData(np.zeros((2, 1), dtype=np.float32))
    dummy.obs_names = ['cell_a', 'cell_b']
    dummy.obsm['spatial'] = np.zeros((2, 2))
    try:
        ccf_coordinates(dummy, 0)
    except ValueError:
        pass
    else:
        raise AssertionError('不应接受只有局部 XY 的输入。')
    table = pd.DataFrame({'slice_id': [0, 0], 'cell_id': ['cell_b', 'cell_a'],
                          'x_ccf': [5., 4.], 'y_ccf': [2., 2.], 'z_ccf': [3., 3.]})
    coords = ccf_coordinates(dummy, 0, table)
    assert np.array_equal(coords[:, 0], [4., 5.])
    return 'PASS：三折划分、缺少 CCF 的拒绝路径、按 cell ID 连接坐标。'

print(input_self_check())

## 5. 加载原始实现与冻结权重

沿用现有源码，不升级到作者新版 API，不改变生成参数。`__main__.TrainConfig` 别名仅用于反序列化作者旧检查点，
不训练也不修改配置。表达模型仍由原 `Inferernce.infer_expression` 创建并以 `eval()` 和 `no_grad()` 推理。

**显存提示：**保持原批量 1500 时，仅一份 500-token FP32 KV cache 约 25.7 GiB；
实现中的缓存复制会继续增加峰值。显存不足会记录错误并停止，不自动减批量、量化或切换精度。

In [ ]:
if not torch.cuda.is_available():
    raise RuntimeError('原设置使用 CUDA；当前内核未检测到 GPU，请先检查环境。')

import __main__
import matplotlib
matplotlib.use('Agg')
import matplotlib.pyplot as plt
from main import TrainConfig as PaperTrainConfig
from models.combined_model import CombinedModel
from inference import Inferernce
from evaluator import Evaluator
from models.biological_model import KDEModelForGuidance

__main__.TrainConfig = PaperTrainConfig

def source_settings_check():
    import ast
    tree = ast.parse((PROJECT / 'inference.py').read_text(encoding='utf-8-sig'))
    calls = [n for n in ast.walk(tree) if isinstance(n, ast.Call)
             and isinstance(n.func, ast.Attribute) and n.func.attr == 'generate_cell_genesis']
    assert len(calls) == 1, '表达生成调用发生变化，请核对原始配置。'
    kw = {k.arg: ast.literal_eval(k.value) for k in calls[0].keywords
          if k.arg in {'batch_size', 'max_new_tokens', 'top_k', 'fast'}}
    assert kw == {'max_new_tokens': 500, 'top_k': 5, 'batch_size': 1500, 'fast': True}, kw
    assert PaperTrainConfig().n_timesteps == 70
    return kw

ORIGINAL_GENERATION = source_settings_check()
model = CombinedModel(location_model_checkpoint=CFG['location_model_checkpoint'],
                      celltype_model_checkpoint=CFG['cluster_model_checkpoint'],
                      gene_exp_model_checkpoint=CFG['expression_model_checkpoint'])
for network in (model.trainer.model, model.celltype_model):
    network.eval()
    for parameter in network.parameters():
        parameter.requires_grad_(False)
evaluator = Evaluator(CFG, metadata_dir=str(METADATA_DIR) + os.sep)
print('位置和类别权重已加载并冻结；表达权重在原推理函数中加载。')
print('保留的表达生成配置:', ORIGINAL_GENERATION)

In [ ]:
def load_geometry(slice_id):
    # AnnData 在内存中读取后，仅把几何信息交给生成管线；表达和真实标签不作为条件。
    a = read_raw(slice_id)
    coords = ccf_coordinates(a, slice_id, sidecar)
    geometry = ad.AnnData(obs=pd.DataFrame(index=a.obs_names.copy()))
    geometry.obsm['aligned_spatial'] = coords[:, [2, 1, 0]]
    return geometry

def evaluation_truth(slice_id):
    # 与原 harmonize_dataset 一致：最大值 > 10 时 log1p，至少 10 个表达基因；不补造未测基因。
    import scanpy as sc
    a = read_raw(slice_id)
    coords = ccf_coordinates(a, slice_id, sidecar)
    a.obsm['aligned_spatial'] = coords[:, [2, 1, 0]]
    if float(a.X.max()) > 10:
        sc.pp.log1p(a)
    sc.pp.filter_cells(a, min_genes=10)
    if not a.n_obs:
        raise ValueError('留出切片预处理后无可评估细胞。')
    return a

def make_identity():
    inputs = {f'raw/merfish_{i}.h5ad': sha256(RAW_DIR / f'merfish_{i}.h5ad') for i in range(5)}
    if CCF_CSV.is_file():
        inputs['ccf_coordinates.csv'] = sha256(CCF_CSV)
    sources = [PROJECT / name for name in
               ['main.py', 'inference.py', 'evaluator.py', 'metrics.py', 'analysis.py', 'data_loader.py', 'gene_exp_model.py']]
    sources += sorted((PROJECT / 'models').rglob('*.py'))
    metadata = [META_PATH, ID_PATH, hierarchy_path, *[METADATA_DIR / f'edges_{c}.pkl' for c in 'xyz']]
    return {'config': CFG, 'holdouts': HOLDOUTS, 'seed': SEED,
            'generation': ORIGINAL_GENERATION, 'input_sha256': inputs,
            'checkpoint_sha256': {k: sha256(v) for k, v in CHECKPOINTS.items()},
            'metadata_sha256': {str(p.relative_to(PROJECT)): sha256(p) for p in metadata},
            'source_sha256': {str(p.relative_to(PROJECT)): sha256(p) for p in sources}}

IDENTITY = make_identity()
stamp = OUTPUT / 'experiment.json'
if stamp.exists():
    if not RESUME or json.loads(stamp.read_text(encoding='utf-8')) != IDENTITY:
        raise ValueError('输出目录已有不同配置/源码/输入/权重，或 RESUME=False；请指定新的 OUTPUT。')
else:
    write_json(stamp, IDENTITY)
write_json(OUTPUT / 'environment.json', {'python': platform.python_version(), 'torch': torch.__version__,
           'cuda': torch.version.cuda, 'gpu': torch.cuda.get_device_name(0),
           'vram_gib': torch.cuda.get_device_properties(0).total_memory / 2**30})

def predict_fold(heldout):
    fold = fold_definition(heldout)
    folder = OUTPUT / f'holdout_{heldout}' / f'seed_{SEED}'
    folder.mkdir(parents=True, exist_ok=True)
    completed = folder / 'completed.json'
    if completed.is_file():
        saved = json.loads(completed.read_text(encoding='utf-8'))
        if sha256(folder / 'prediction.h5ad') != saved['prediction_sha256']:
            raise ValueError(f'已有预测被修改：{folder}')
        if sha256(folder / 'metrics.json') != saved['metrics_sha256']:
            raise ValueError(f'已有指标被修改：{folder}')
        return json.loads((folder / 'metrics.json').read_text(encoding='utf-8'))

    np.random.seed(SEED)
    torch.manual_seed(SEED)
    torch.cuda.manual_seed_all(SEED)
    references = [load_geometry(i) for i in fold['reference']]
    target_geometry = load_geometry(heldout)
    # 原入口需要两张参考，并选择距离排序的第二张拟合 KDE；保留这一实际实现。
    dists = [(a.obsm['aligned_spatial'][:, -1].mean()
              - target_geometry.obsm['aligned_spatial'][:, -1].mean()) ** 2 for a in references]
    kde_reference = references[np.argsort(dists)[1]].copy()
    kde_reference.obsm['aligned_spatial'] = kde_reference.obsm['aligned_spatial'][:, :2]
    kde = KDEModelForGuidance([kde_reference], bandwidth=CFG['kde_bandwidth'])
    kde.fit()

    token_map = SimpleNamespace(get_label_from_token=lambda tokens: [ID_TO_LABEL[int(t)] for t in tokens])
    loader = SimpleNamespace(metadata_dir=str(METADATA_DIR) + os.sep,
                             gene_exp_model=token_map, test_slices=[target_geometry],
                             reference_slices=references, train_slices=references)
    cfg = copy.deepcopy(CFG)
    cfg.update(artifact_dir=str(folder), slice_index=heldout)
    inference = Inferernce(model, kde, loader, cfg)
    started = time.perf_counter()
    torch.cuda.reset_peak_memory_stats()
    try:
        with torch.no_grad():
            # 复用三个原始阶段，绕过 run_inference 中依赖真实 cluster 的绘图。
            prediction = ad.AnnData(obs=target_geometry.obs.copy())
            prediction = inference.infer_location(prediction, target_geometry)
            prediction = inference.infer_cluster(prediction, target_geometry)
            prediction, generated_tokens = inference.infer_expression(prediction, target_geometry)
        del generated_tokens
        prediction.obs['cluster'] = [ID_TO_LABEL[int(t)] for t in prediction.obs['token']]
        prediction.obs['slice_id'] = heldout
        coords = np.asarray(prediction.obsm['spatial'])
        prediction.obs['x_ccf'], prediction.obs['y_ccf'], prediction.obs['z_ccf'] = coords[:, 2], coords[:, 1], coords[:, 0]
        prediction.obsm['spatial_ccf_mm'] = coords.copy()
        prediction.uns['protocol'] = 'frozen author checkpoints; target geometry/count available; no training'
        prediction.write_h5ad(folder / 'prediction.h5ad', compression='gzip')

        truth = evaluation_truth(heldout)
        overlap = [g for g in truth.var_names if g in set(META['gene_set'])]
        pd.Series(overlap, name='gene').to_csv(folder / 'evaluated_gene_panel.csv', index=False)
        # Evaluator 原地切换到二维，使用副本避免更改已经保存的三维预测。
        result = evaluator.evaluate(prediction.copy(), truth, sample=CFG['metric_sampling'])
        metrics = {k: float(v) for k, v in result.items()}
        if not metrics or not all(np.isfinite(list(metrics.values()))):
            raise ValueError('评估包含非有限数值或没有指标；该折不标记为完成。')
        metrics.update(heldout=heldout, seed=SEED, n_predicted=prediction.n_obs,
                       n_evaluated=truth.n_obs, n_measured_overlap_genes=len(overlap),
                       elapsed_seconds=time.perf_counter() - started,
                       peak_allocated_gib=torch.cuda.max_memory_allocated() / 2**30)
        write_json(folder / 'config.json', {**cfg, 'fold': fold})
        write_json(folder / 'metrics.json', metrics)
        write_json(completed, {'prediction_sha256': sha256(folder / 'prediction.h5ad'),
                               'metrics_sha256': sha256(folder / 'metrics.json')})
        return metrics
    except Exception as exc:
        write_json(folder / 'failed.json', {'heldout': heldout, 'error': repr(exc),
                   'note': '未自动降低批量、精度或生成长度；此折未完成。'})
        raise
    finally:
        plt.close('all')
        gc.collect()
        torch.cuda.empty_cache()

## 6. 依次完成三折预测与原指标评估

下一个单元会运行真实预测，可能耗时较长。遇到 OOM 或输入错误会停止，不用其他折或占位数代替。
重新运行时仅跳过与本次身份信息一致且文件校验通过的完整结果。模型没有任何优化器更新。

In [ ]:
records = []
for heldout in HOLDOUTS:
    print(f'\n===== 留出 merfish_{heldout}；参考 {fold_definition(heldout)["reference"]} =====', flush=True)
    records.append(predict_fold(heldout))
    pd.DataFrame(records).to_csv(OUTPUT / 'per_fold.csv', index=False)

assert len(records) == 3 and {r['heldout'] for r in records} == set(HOLDOUTS)
if {k: sha256(v) for k, v in CHECKPOINTS.items()} != IDENTITY['checkpoint_sha256']:
    raise RuntimeError('运行后权重文件哈希发生变化。')
per_fold = pd.DataFrame(records).sort_values('heldout')
metric_cols = [c for c in per_fold if c.startswith(('soft_spearman_', 'soft_f1_'))]
summary = per_fold[metric_cols].agg(['mean', 'std']).T
summary.columns = ['three_fold_mean', 'three_fold_sample_std']
summary['n_folds'] = 3
summary.to_csv(OUTPUT / 'summary.csv', index_label='metric')
display(per_fold)
display(summary)
print('三折均已完成，原权重文件哈希未改变。输出：', OUTPUT)

## 结果位置与解释

```text
results/merfish_pretrained_3fold/
├── input_audit.csv                 # 输入审查；仅有此文件不代表推理完成
├── experiment.json                # 配置、原参数与输入/源码/权重哈希
├── environment.json
├── holdout_1/seed_42/              # holdout_2、holdout_3 同结构
│   ├── prediction.h5ad            # 2000 基因预测、作者 cluster、CCF 三维坐标
│   ├── evaluated_gene_panel.csv   # 实测与模型基因交集；原指标内部仍应用自己的表达过滤
│   ├── config.json
│   ├── metrics.json
│   └── completed.json             # 成功后才生成；失败时可能只有 failed.json
├── per_fold.csv
└── summary.csv                    # 三折指标均值与样本标准差；不是多 seed 方差
```

- 表达 radii 与原 Evaluator 相同：0.03、0.04、0.05、0.09、0.12、0.15 mm。
- 原始数据最大表达值大于 10 时采用 log1p，与原仓库 harmonize 分支一致；不训练新的预处理器。
- 预测的 2000 基因中，只有实测交集能得到本数据集的直接评估，未测基因没有真实评估值。
- 直接使用现有权重跨数据集推理，性能不保证；输入通过数值检查也不代表已验证图谱配准正确。
- 若数据没有 CCF：先补真实配准，再继续第 3 节；不要通过关闭检查或补造坐标生成“复现分数”。

方法来源：[论文作者代码与 CCF 输入说明](https://github.com/gkrieg/mimyr#python-api)。
本 notebook 实际执行的是本地旧版仓库源码和已有权重，不混用新版默认设置。